In [11]:
import duckdb
import polars as pl
import plotly.express as px
import plotly.graph_objects as go

# Connect to DuckDB
conn = duckdb.connect("../data/mbta.duckdb", read_only=True)

# Helper to query into polars
def query(sql: str) -> pl.DataFrame:
    return conn.execute(sql).pl()

In [12]:
routes = query("""
    SELECT route_id, long_name, route_type, route_type_desc, fare_class, line_id
    FROM staging.stg_routes
    ORDER BY route_type, long_name
""")

print(f"Total routes: {len(routes)}")
print(f"\nRoutes by type:")
print(routes.group_by("route_type_desc").agg(pl.count().alias("count")).sort("count", descending=True))

Total routes: 176

Routes by type:
shape: (5, 2)
┌─────────────────┬───────┐
│ route_type_desc ┆ count │
│ ---             ┆ ---   │
│ str             ┆ u32   │
╞═════════════════╪═══════╡
│ Bus             ┆ 149   │
│ Commuter Rail   ┆ 13    │
│ Ferry           ┆ 6     │
│ Light Rail      ┆ 5     │
│ Heavy Rail      ┆ 3     │
└─────────────────┴───────┘


/var/folders/3w/csvlpk454tq67v7qllhl7ghm0000gr/T/ipykernel_38314/3142220740.py:9: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  print(routes.group_by("route_type_desc").agg(pl.count().alias("count")).sort("count", descending=True))


In [13]:
route_counts = routes.group_by("route_type_desc").agg(
    pl.count().alias("count")
).sort("count", descending=True).to_pandas()

fig = px.bar(
    route_counts,
    x="route_type_desc",
    y="count",
    title="MBTA Routes by Type",
    color="route_type_desc",
    text="count",
)
fig.update_layout(showlegend=False, xaxis_title="Route Type", yaxis_title="Number of Routes")
fig.show()

/var/folders/3w/csvlpk454tq67v7qllhl7ghm0000gr/T/ipykernel_38314/4196429111.py:2: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  pl.count().alias("count")


In [14]:
stops = query("""
    SELECT stop_id, stop_name, municipality, latitude, longitude,
           location_type, location_type_desc, vehicle_type_desc, zone_id
    FROM staging.stg_stops
""")

print(f"Total stops: {len(stops)}")
print(f"\nStops by location type:")
print(stops.group_by("location_type_desc").agg(pl.count().alias("count")).sort("count", descending=True))
print(f"\nStops by municipality (top 15):")
print(
    stops.group_by("municipality").agg(pl.count().alias("count"))
    .sort("count", descending=True)
    .head(15)
)

/var/folders/3w/csvlpk454tq67v7qllhl7ghm0000gr/T/ipykernel_38314/1688227727.py:9: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  print(stops.group_by("location_type_desc").agg(pl.count().alias("count")).sort("count", descending=True))


Total stops: 9433

Stops by location type:
shape: (4, 2)
┌────────────────────┬───────┐
│ location_type_desc ┆ count │
│ ---                ┆ ---   │
│ str                ┆ u32   │
╞════════════════════╪═══════╡
│ Stop/Platform      ┆ 7807  │
│ Generic Node       ┆ 1028  │
│ Entrance/Exit      ┆ 323   │
│ Station            ┆ 275   │
└────────────────────┴───────┘

Stops by municipality (top 15):
shape: (15, 2)
┌──────────────┬───────┐
│ municipality ┆ count │
│ ---          ┆ ---   │
│ str          ┆ u32   │
╞══════════════╪═══════╡
│ Boston       ┆ 2920  │
│ Quincy       ┆ 569   │
│ Cambridge    ┆ 506   │
│ Newton       ┆ 378   │
│ Lynn         ┆ 345   │
│ …            ┆ …     │
│ Waltham      ┆ 194   │
│ Braintree    ┆ 181   │
│ Milton       ┆ 177   │
│ Arlington    ┆ 157   │
│ Weymouth     ┆ 150   │
└──────────────┴───────┘


/var/folders/3w/csvlpk454tq67v7qllhl7ghm0000gr/T/ipykernel_38314/1688227727.py:12: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  stops.group_by("municipality").agg(pl.count().alias("count"))


In [15]:
stops_pd = stops.filter(pl.col("location_type") == 1).to_pandas()

# Stations don't have vehicle_type — color by zone instead
fig = px.scatter_map(
    stops_pd,
    lat="latitude",
    lon="longitude",
    hover_name="stop_name",
    hover_data=["municipality", "zone_id"],
    color="zone_id",
    title="MBTA Stations",
    zoom=10,
    height=600,
)
fig.update_layout(map_style="carto-positron")
fig.show()

In [16]:
muni_counts = (
    stops.group_by("municipality")
    .agg(pl.count().alias("stop_count"))
    .sort("stop_count", descending=True)
    .head(20)
    .to_pandas()
)

fig = px.bar(
    muni_counts,
    x="municipality",
    y="stop_count",
    title="Top 20 Municipalities by Number of Stops",
    text="stop_count",
)
fig.update_layout(xaxis_tickangle=-45)
fig.show()

/var/folders/3w/csvlpk454tq67v7qllhl7ghm0000gr/T/ipykernel_38314/2853638623.py:3: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  .agg(pl.count().alias("stop_count"))


In [17]:
# Schedules reference platform-level stops, not stations
# Join through parent_station_id to get station coordinates
subway_stops = query("""
    SELECT DISTINCT
        coalesce(s.parent_station_id, s.stop_id) as station_id,
        coalesce(parent.stop_name, s.stop_name) as stop_name,
        coalesce(parent.municipality, s.municipality) as municipality,
        coalesce(parent.latitude, s.latitude) as latitude,
        coalesce(parent.longitude, s.longitude) as longitude,
        r.long_name as route_name,
        '#' || r.color as route_color,
        r.route_type_desc
    FROM staging.stg_schedules sch
    JOIN staging.stg_stops s ON sch.stop_id = s.stop_id
    LEFT JOIN staging.stg_stops parent ON s.parent_station_id = parent.stop_id
    JOIN staging.stg_routes r ON sch.route_id = r.route_id
    WHERE r.route_type IN (0, 1)
""").to_pandas()

print(f"Subway stops found: {len(subway_stops)}")
print(f"Routes: {subway_stops['route_name'].unique()}")

if len(subway_stops) > 0:
    fig = px.scatter_map(
        subway_stops,
        lat="latitude",
        lon="longitude",
        hover_name="stop_name",
        hover_data=["municipality", "route_name"],
        color="route_name",
        title="MBTA Subway Station Network",
        zoom=11,
        height=600,
    )
    fig.update_layout(map_style="carto-positron")
    fig.show()
else:
    print("No subway stops found — check schedule/stop join")

Subway stops found: 165
Routes: ['Green Line B' 'Green Line C' 'Blue Line' 'Green Line D' 'Orange Line'
 'Red Line' 'Green Line E']


In [18]:
print("=" * 50)
print("MBTA NETWORK SUMMARY")
print("=" * 50)

total_routes = len(routes)
subway_routes = len(routes.filter(pl.col("route_type").is_in([0, 1])))
bus_routes = len(routes.filter(pl.col("route_type") == 3))
total_stops = len(stops)
stations = len(stops.filter(pl.col("location_type") == 1))
municipalities = stops["municipality"].n_unique()

print(f"Total routes:        {total_routes}")
print(f"  Subway/Light Rail: {subway_routes}")
print(f"  Bus:               {bus_routes}")
print(f"Total stops:         {total_stops}")
print(f"  Stations:          {stations}")
print(f"Municipalities:      {municipalities}")

conn.close()

MBTA NETWORK SUMMARY
Total routes:        176
  Subway/Light Rail: 8
  Bus:               149
Total stops:         9433
  Stations:          275
Municipalities:      110
